# Transformer From Scratch

Practice Session by Amodh Herath with parallel to DataCamp Transfomers with Pytorch and Attention is all you need paper. Here a transformer architecture is built from scratch using pytorch for the learning and understanding of this architecture.

In [1]:
import torch
import math
import torch.nn as nn

f:\Projects\transformer\.transformerenv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Input Embedding

In [4]:


class InputEmbedding(nn.Module):
    def __init__(self,vocab_size:int , d_model:int) -> None:
        super().__init__()  
        self.d_model=d_model
        self.vocab_size=vocab_size
        self.embedding=nn.Embedding(num_embeddings=vocab_size,embedding_dim=d_model)  
    
    def forward(self,x):
        return self.embedding(x)*math.sqrt(self.d_model)

## Positional Encoding


Generate the positional Encoding for tokens based on its position by using sin and cosine values.
-   even (2i) --> Sin(pos/1000^(2i/d_model))
-   odd (2i+1) --> Cos(pos/1000^(2i/d_model))

In [ ]:
class PositionalEncoding(nn.Module):
    pe: torch.Tensor
    def __init__(self, d_model, max_seq_len) -> None:
        super().__init__()
        # Initial 1D tensor filled with zeroes
        pe=torch.zeros(size=(max_seq_len,d_model))

        # position of the token from (0,...max_sequence_length)
        position=torch.arange(start=0 , end=max_seq_len).unsqueeze(1)
        

        '''
        Calculate the divisional term (1000^(2i/d_model)), it was translated to exponential base for efficient computations.
        the denominators  are identical for every pair of adjacent indices (2i and 2i+1).there are 
        only d_model/2 unique frequencies so we can use step=2
        how this would look like --> 0,2,4,6,...d_model-2
        
        in case of 2i absolute last number in this sequence, we substitute the maximum possible value of 
        2i (which we established is 2*(d_model/2 -1 )
        2i --> 0,2,4,6,8...d_model-2
        '''
        
        div_term=torch.exp(torch.arange(start=0 , end=d_model , step=2, dtype=torch.float)*-math.log(10000)/d_model)
        
        # all rows and all even columns
        pe[:,0::2] = torch.sin(position*div_term)
        # all rows and all odd columns
        pe[:,1::2] = torch.cos(position*div_term)
        
        # add this as a non trainable paramter but part of model state
        self.register_buffer('pe',pe.unsqueeze(0))
        
    def forward(self,x):
        return x + self.pe[:,:x.size(0)]

## MultiHead Attention

In [9]:
import torch.nn.functional as F

In [ ]:
class MultiHeadAttention(nn.Module):
   def __init__(self, d_model: int, num_heads: int):
      super().__init__()
      # check whether d_model is divisible by num_heads
      assert d_model % num_heads == 0
      self.num_heads = num_heads
      self.d_model = d_model
      # Actual size of the vector that 1 head looks at
      self.head_dim = d_model // num_heads
      
      ''' The Input size d_model(512 as an example) ensures that every single head has access to the complete, rich context of the entire 
      token embedding. The Output size d_model(512) is simply the sum total 
      of all the heads' individual outputs packed together.
      Learnable weight matrices belong to all heads combined for the Query, Key and Value.
      '''
      # Query - what the token is searching for
      self.query_linear = nn.Linear(d_model, d_model, bias=False)
      # current contents of the token
      self.key_linear = nn.Linear(d_model, d_model, bias=False)
      # what the actual information about the token available
      self.value_linear = nn.Linear(d_model, d_model, bias=False)
      
      # concatenate all the outputs of the attention heads 
      self.final_concat_output = nn.Linear(d_model, d_model)

   ''' break apart separate dimensions for each individual attention head '''
   def split_heads(self, x: torch.Tensor, batch_size: int):
      # [batch_size, sequence_length, d_model]
      seq_length = x.size(1)
      # split the d_model into num_of_heads x head_dim
      x = x.reshape(batch_size, seq_length, self.num_heads, self.head_dim)
      # shuffle the seq_len and the num_heads axes so that we get [batch_size, num_heads, seq_len, head_dim]
      return x.permute(0, 2, 1, 3)
   
   def compute_attention(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, d_model: int, num_heads: int, mask=None):
      # Get the dot product of the query and the key matrices 
      # b = batch, h = num_of_heads, i = query sequence, j = key sequence, d = head_dimension
      scores = torch.matmul(query, key.transpose(-2, -1)) / (self.head_dim ** 0.5)
      # alternative -> Einstein Summation (operation, tensors):
      # scores = torch.einsum('bhid,bhjd->bhij', query, key) / (self.head_dim ** 0.5)
      if mask is not None:
         scores = scores.masked_fill(mask == 0, float('-inf'))
      
      attention_weights = torch.nn.functional.softmax(scores, dim=-1)
      attention_output = torch.matmul(attention_weights, value)
      return attention_output
    
   def combine_heads(self, x: torch.Tensor, batch_size: int) -> torch.Tensor:
      # reverse of the split heads which makes the num heads and head dim --> model_dim
      x = x.permute(0, 2, 1, 3).contiguous()  # [batch_size, seq_len, num_heads, head_dim] 
      return x.view(batch_size, -1, self.d_model)
      
   def forward(self, query: torch.Tensor, key: torch.Tensor, value: torch.Tensor, mask=None):
      # [batch_size, num_heads, seq_len, head_dim]
      batch_size = query.size(0)
      # splitting the queries to the heads
      query = self.split_heads(self.query_linear(query), batch_size=batch_size)
      key = self.split_heads(self.key_linear(key), batch_size=batch_size)
      value = self.split_heads(self.value_linear(value), batch_size=batch_size)
      
      # Compute scaled dot-product attention independently across all heads in parallel.
      # Each head attends to different relational features and dependencies in the sequence.
      attention_weights = self.compute_attention(query=query, 
                                                 key=key, 
                                                 value=value, 
                                                 d_model=self.d_model, 
                                                 num_heads=self.num_heads, 
                                                 mask=mask)
      
      # (concatenate) the separate head outputs back into a single vector per token
      # of shape [batch_size, seq_len, d_model], combining the diverse contexts learned by each head.
      output = self.combine_heads(attention_weights, batch_size)
      
      # Apply a final linear projection to linearly mix and integrate the concatenated 
      # features from all heads, allowing the network to learn unified representations from the combined subspaces.
      return self.final_concat_output(output)